# Practice 01: Robot models and MuJoCo basics

**Goal:** inspect real robot models, connect their equations to simulator quantities, and understand MuJoCo's state and simulation pipeline.

**Before the practice:** read [lecture 01: Robotic systems, models, and state space](https://simeon-ned.github.io/forc/notes/01-modeling/). The web lecture explains the equations used below; the [lecture slides](https://simeon-ned.github.io/forc/lectures/01-modeling/) provide the classroom outline.

**Models from MuJoCo Menagerie**

- **UR5e:** fixed-base kinematics, dynamics, and energy.
- **Unitree G1:** humanoid with a floating base and foot contacts.
- **Unitree Go2:** quadruped with a floating base, contact forces, and passive motion.

**Learning path**

1. Load and render the UR5e before inspecting its parameters.
2. Explore state, coordinates, body and COM kinematics.
3. Check dynamics, energy, and external forces.
4. Load and render G1 and Go2, then inspect their floating bases and contacts.
5. Advance a passive simulation and edit a model with `MjSpec`.

**How to use the notebook**

- Run cells from top to bottom and read each explanation before its code.
- Treat assertions as checks of the equations and array conventions.
- Model queries use separate `MjData` objects to preserve other experiments.

Feedback fundamentals are covered in [practice 02](../p02-control-basics/tutorial.ipynb), followed by manipulator control and linearization in [practice 03](../p03-manipulator/control_and_linearization.ipynb).

## Setup

### Choose your environment

Run these commands **from the repository root**, then select that environment's notebook kernel.

**With uv:**

```bash
uv sync --locked --extra p01
uv run --locked --extra p01 jupyter lab practices/p01-simulation/tutorial.ipynb
```

**With conda:**

```bash
conda env create --file environment.yml
conda activate forc
python scripts/setup_conda.py p01
python -m jupyterlab practices/p01-simulation/tutorial.ipynb
```

See the [shared setup guide](../README.md) for editor kernels and system libraries. Dependencies are declared in `pyproject.toml` and pinned in `uv.lock`; do not install packages from notebook cells.

### Model downloads

[MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie) supplies the robot's XML, meshes, and license.

- **First load:** `mm.load(name)` downloads the requested model and compiles its scene.
- **Later loads:** the local cache is reused, including offline.
- **Cache location on Linux:** `~/.cache/mujoco_menagerie`. Override it with `MENAGERIE_CACHE_DIR`.

To prepare for an offline class:

```bash
uv run --locked --extra p01 mujoco-menagerie prefetch universal_robots_ur5e unitree_g1 unitree_go2
```

### Graphics and video

- **Desktop:** the default OpenGL backend usually works.
- **Headless Linux:** set `MUJOCO_GL=egl`, or use `osmesa` with its system libraries installed.
- **Timing:** choose the backend **before importing MuJoCo**. Restart the kernel after changing it.
- **Video encoding:** FFmpeg must be available on the system PATH.

### Import the tools

- `mujoco`: model compilation and physics.
- `mujoco_menagerie`: robot retrieval and caching.
- `numpy`: arrays and numerical calculations.
- `mediapy` and `matplotlib`: images, videos, and plots.

A seeded random generator makes the numerical examples repeatable. Importing Menagerie alone does not download a model.

In [ ]:
import mujoco
import mujoco_menagerie as mm
import mediapy as media
import numpy as np
import matplotlib.pyplot as plt

renderer = None
rng = np.random.default_rng(0)
print("MuJoCo:", mujoco.__version__)

## 1. Load and view the robot

### Compile the scene and create its state

An MJCF scene describes bodies, joints, inertias, collision shapes, and actuators. The Menagerie scene also supplies the floor and lighting.

The next cell performs three steps:

1. `mm.load(...)` resolves the files and returns a compiled `MjModel`.
2. `MjData(model)` allocates a separate state and computation workspace.
3. `mj_forward(...)` initializes derived quantities **without advancing time**.

In [ ]:
model = mm.load("universal_robots_ur5e")
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
print(f"nq={model.nq}, nv={model.nv}, nu={model.nu}")

### First look: the loaded UR5e

**Before doing calculations, inspect the robot you loaded.**

1. Select the model's first keyframe when one is available; otherwise use its default pose.
2. Refresh the body poses with `mj_forward`.
3. Create an offscreen renderer and frame the scene with a camera.
4. Display the robot without advancing simulation time.

The 800 × 600 image uses the model's geometry, materials, floor, and lighting. We will reuse this renderer for later images and videos.

In [ ]:
if renderer is not None:
    renderer.close()
if model.nkey:
    mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)

model.vis.global_.offwidth = 800
model.vis.global_.offheight = 600
renderer = mujoco.Renderer(model, width=800, height=600)
preview_camera = mujoco.MjvCamera()
preview_camera.lookat[:] = model.stat.center
preview_camera.distance = 1.8 * model.stat.extent
preview_camera.azimuth = 135
preview_camera.elevation = -20
renderer.update_scene(data, camera=preview_camera)
robot_preview = renderer.render()
media.show_image(robot_preview)
print("UR5e scene loaded; simulated time:", data.time, "s")

## 2. Model, state, and coordinates

### Model parameters: `MjModel`

`MjModel` contains the compiled model description. Its parameters stay fixed during normal simulation, unless you deliberately edit them.

Useful fields include:

- `nbody`: number of bodies, including the world.
- `ngeom`: number of visual and collision shapes.
- `geom_rgba`: colors of those shapes.

The [`mjModel` reference](https://mujoco.readthedocs.io/en/stable/APIreference/APItypes.html#mjmodel) describes the complete structure.

### Count positions, velocities, and controls

- `nq`: number of stored position coordinates.
- `nv`: number of velocity degrees of freedom.
- `nu`: number of control inputs.

For the fixed-base UR5e:

$$
n_q=n_v=n_u=6.
$$

A free base adds **seven** position entries but only **six** velocity entries. We will examine that difference below.

In [ ]:
nbody = model.nbody
nv = model.nv
nq = model.nq
print(f'Number of bodies: {nbody}')
print(f'Total number of DoFs in the model: {nv}')
print(f'Total number of generalized coordinates: {nq}')
print(f'Number of geoms in the scene: {model.ngeom}')
print(f'Colors of each geom: \n {model.geom_rgba}')

Named access avoids depending on the order of bodies in an XML file. List the names first, including the world body.

In [ ]:
[model.body(i).name for i in range(model.nbody)]

**Compare model parameters with state values:**

- `model.body(name)`: mass, inertia, and other model parameters.
- `data.body(name)`: quantities such as the body's current world pose.

Use the same body name to query the appropriate structure.

In [ ]:
model.body('upper_arm_link') # model.body(3)

### Coordinates, joint limits, and addresses

The model stores configuration, generalized velocity, and control:

$$
q\in\mathcal Q,\qquad v\in\mathbb R^{n_v},\qquad u\in\mathbb R^{n_u}.
$$

In general, their relationship is

$$
\dot q=W(q)v.
$$

For the UR5e's hinge joints, velocity equals the time derivative of the joint angles. A quaternion instead uses four position numbers for three tangent velocity components.

**Read the joint arrays carefully:**

- `jnt_qposadr`: start of a joint's position entries.
- `jnt_dofadr`: start of its velocity entries.
- `jnt_range`: limits, meaningful when `jnt_limited` is true.
- Hinge limits use **radians**; slide limits use **metres**.

**Next cell:** list joint names, addresses, and limits. A joint ID is not necessarily its position or velocity address.

In [ ]:
for joint_id in range(model.njnt):
    joint = model.joint(joint_id)
    limit = model.jnt_range[joint_id] if model.jnt_limited[joint_id] else "unlimited"
    print(joint.name, "qpos address:", model.jnt_qposadr[joint_id],
          "velocity address:", model.jnt_dofadr[joint_id], "limits:", limit)

assert model.nq == model.nv == model.nu == 6

### Changing state: `MjData`

`MjData` holds the simulation state and intermediate results:

- `qpos`, `qvel`: positions and velocities.
- `time`: simulated time.
- `xpos`, `xquat`: derived body poses.
- Force and acceleration arrays: results of the dynamics calculations.

Changing a state array does not automatically refresh every derived field. Run the appropriate forward computation before reading those fields.

`data.time` is simulated time in seconds. Creating data or running forward kinematics does not advance it; integration with `mj_step` does.

In [ ]:
print(f'time in simulator {data.time}')

`qpos` stores generalized positions. For this arm every joint is a hinge, so its six entries are angles in radians.

In [ ]:
print(f'coordinates:\n {data.qpos}')

`xpos` stores body-frame origins in world coordinates (metres). It is a derived quantity: refresh the forward pipeline after changing `qpos` before reading it.

In [ ]:
print(f'positions:\n {data.xpos}')

**Next cell:**

1. Assign a reproducible joint configuration.
2. Call `mj_forward` to refresh poses and dynamics quantities.
3. Read the updated world positions.

This evaluates one state; it does **not** integrate time.

In [ ]:
data.qpos = rng.uniform(-1.0, 1.0, model.nq) 
mujoco.mj_forward(model, data)
print(f'random positions:\n {data.xpos}')

**Interpret the arrays:**

- `qvel`: six hinge angular velocities, in **rad/s**.
- `xquat`: body orientations as scalar-first unit quaternions `(w, x, y, z)`.

Changing position alone does not create velocity. A quaternion uses **four** numbers to represent an orientation with **three** rotational degrees of freedom.

In [ ]:
print(f'Joint velocities: {data.qvel}')
print(f'Quaternions: \n {data.xquat}')

### Simulation settings: `MjOption`

The options object is stored in `model.opt` and corresponds to the MJCF `<option>` element.

Inspect the numerical and physical settings:

- Gravity.
- Timestep.
- Integrator.
- Constraint-solver settings.

Record these settings when comparing experiments: they can affect both stability and accuracy.

In [ ]:
model.opt

## 3. Render a pose and animate motion

We will reuse the renderer created immediately after loading the robot.

### Reset the state

1. `mj_resetData` restores the model's default state and clears time, controls, and applied forces.
2. `mj_forward` refreshes derived quantities before we draw the pose.

In [ ]:
mujoco.mj_resetData(model, data)  # Reset state and time.
mujoco.mj_forward(model, data)

### Prescribe a pose

The following sequence changes what we see:

1. Assign the desired joint angles to `qpos`.
2. Refresh body poses with `mj_forward`.
3. Build the visual scene with `renderer.update_scene`.
4. Display the RGB image returned by `renderer.render`.

This is a prescribed configuration, not a simulated movement.

In [ ]:
q0 = np.array([-0.6,-1.2,-2.1,0.7,0.7,0.7])
data.qpos = q0 
mujoco.mj_forward(model, data)
renderer.update_scene(data)
media.show_image(renderer.render())

### Create a kinematic animation

The first joint follows a sine wave:

$$
q_1(t)=-3+2\sin(t).
$$

- Each frame uses a directly assigned configuration.
- `mj_forward` updates the corresponding poses without time integration.
- The frame rate controls image sampling, independently of a physics timestep.

**Interpretation:** this illustrates a motion, but does not test actuator effort, contact forces, or physical feasibility.

In [ ]:
timespan = np.linspace(0, 2, 500)
# print(timespan)
framerate = 30  # (Hz)

# Simulate and display video.
frames = []

for time in timespan:
    data.qpos[0] = -3+2*np.sin(time)
    mujoco.mj_forward(model, data)
    if len(frames) < time * framerate:
        mujoco.mj_forward(model, data)
        renderer.update_scene(data)
        pixels = renderer.render()
        frames.append(pixels)

### Display the frames

`show_video` encodes the RGB images at the selected display rate.

$$
\text{video duration}=\frac{\text{number of frames}}{\text{frames per second}}.
$$

In [ ]:
media.show_video(frames, fps=framerate)

## 4. Kinematics

Reset before the kinematics experiment so previous animation poses and controls cannot affect it.

In [ ]:
mujoco.mj_resetData(model, data)  # Reset state and time.

Set a known configuration and refresh the forward pipeline. All poses and Jacobians below must refer to this same configuration.

In [ ]:
data.qpos = np.array([-0.6,-1.2,-2.1,0.7,0.7,0.7])
mujoco.mj_forward(model, data)

#### Forward kinematics

Forward kinematics maps joint positions to body poses. Each row of `xpos` is the world position of a body origin; the first row belongs to the world.

In [ ]:
print(f'positions:\n {data.xpos}')

Each `xquat` row is a scalar-first unit quaternion rotating a body-local frame into the world frame. Quaternions avoid the singularities of Euler-angle parameterizations.

In [ ]:
print(f'quaternions of body frames:\n {data.xquat}')

`xmat` stores the same body-to-world rotations as flattened 3 × 3 matrices. Reshape a row to `(3, 3)` to transform a body-local vector into world coordinates.

In [ ]:
print(f'rotation matrices of body frames:\n {data.xmat}')

**Distinguish body origins, COMs, and joints:**

- `xipos`: body centers of mass in world coordinates.
- `ximat`: orientations of the inertial frames.
- `xanchor`: joint anchor positions in the world frame.
- `xaxis`: joint axes in the world frame.

A body's center of mass generally differs from its frame origin.

In [ ]:
print(f'positions of body COMs :\n {data.xipos}')  
print(f'orientations aligned with inertia:\n {data.ximat}')   
print(f'positions:\n {data.xanchor}')  
print(f'positions:\n {data.xaxis}')

#### Jacobians

### Point and angular Jacobians

At a point fixed to a body, the Jacobians map generalized velocity to Cartesian velocity:

$$
v_p=J_p(q)v,\qquad \omega=J_r(q)v.
$$

For this hinge-only arm, generalized velocity is the derivative of joint position.

**Conventions for `mj_jac`:**

- Both output arrays have shape `(3, nv)`.
- Linear and angular velocities are expressed in **world axes**.
- The point argument is its current **world position**.
- The body ID identifies the body to which the point is attached.

**Next cell:** refresh the forward pipeline without stepping, then compute both Jacobians at one fixed configuration.

In [ ]:
jacp = np.zeros((3, model.nv))
jacr = np.zeros((3, model.nv))
mujoco.mj_forward(model, data)
body_id = model.body("upper_arm_link").id
point = data.xpos[body_id].copy()
mujoco.mj_jac(model, data, jacp, jacr, point, body_id)
jacp_point, jacr_point = jacp.copy(), jacr.copy()

`mj_jacBody` selects the body origin automatically. Because the previous cell chose that same point, its Jacobian should agree with `mj_jac`.

In [ ]:
mujoco.mj_jacBody(model, data, jacp, jacr, body_id)
np.testing.assert_allclose(jacp, jacp_point)
np.testing.assert_allclose(jacr, jacr_point)

Inspect the translational Jacobian: column j gives the world linear velocity caused by unit generalized velocity j with all others zero.

In [ ]:
jacp

Inspect the angular Jacobian: column j gives the world angular velocity caused by unit generalized velocity j.

In [ ]:
jacr

Compute a Jacobian for every body at one fixed pose. This is a batch of instantaneous kinematic queries, so use `mj_forward`, not `mj_step`, before the loop.

In [ ]:
data.qpos = np.array([-0.6,-1.2,-2.1,0.7,0.7,0.7])
jacp_body = np.zeros((model.nbody, 3, model.nv))
jacr_body = np.zeros((model.nbody, 3, model.nv))
mujoco.mj_forward(model, data)
q = data.qpos.copy()
for body_id in range(model.nbody):
    mujoco.mj_jacBody(model, data, 
                      jacp_body[body_id], 
                      jacr_body[body_id], body_id)

#### Velocities

### Predict body velocities

Multiply each body's Jacobian by a chosen joint-velocity vector:

$$
v_i=J_{p,i}v,\qquad \omega_i=J_{r,i}v.
$$

- Both results use world axes.
- Linear velocity is measured in **m/s**; angular velocity in **rad/s**.
- This calculation does not modify `data.qvel` or run a simulation.

The example uses 20 rad/s to make the numerical effect visible; it is an illustrative input, not an operating recommendation.

In [ ]:
vel_world = np.zeros((model.nbody, 3))
omega_world = np.zeros((model.nbody, 3))

dq = 20*np.ones(model.nv)

for body_id in range(model.nbody):
    vel_world[body_id,:] = jacp_body[body_id]@dq
    omega_world[body_id,:] = jacr_body[body_id]@dq

The angular velocity predictions have units rad/s and are expressed in world axes.

In [ ]:
omega_world

The linear velocity predictions have units m/s and are expressed in world axes.

In [ ]:
vel_world

Reinspect the angular velocities alongside the linear velocities; they describe different parts of each body’s instantaneous motion.

In [ ]:
omega_world

### Local axes, world axes, and acceleration

Let the rotation matrix map local vectors into world axes. At the **same physical point**, the Jacobians transform as

$$
J_{p,\mathrm{local}}=R^T J_{p,\mathrm{world}},\qquad
J_{r,\mathrm{local}}=R^T J_{r,\mathrm{world}}.
$$

Changing the reference **origin** also requires a translation term. Rotating components alone is sufficient only when the point remains the same.

**Acceleration in world axes**

$$
a_p=J_p\dot v+\dot J_pv,
$$

$$
\alpha=J_r\dot v+\dot J_rv.
$$

`mj_jacDot` evaluates the Jacobian derivative along the current `qvel`.

**Next cell:**

1. Create separate state data with a prescribed velocity and acceleration.
2. Compare the velocity expressed in local and world axes.
3. Evaluate the acceleration formula.
4. Check it against a central difference of point velocity.

For a local-axis Jacobian derivative, the changing rotation matrix must also be differentiated.

In [ ]:
kin_data = mujoco.MjData(model)
kin_data.qpos[:] = q0
kin_data.qvel[:] = np.linspace(-0.2, 0.3, model.nv)
mujoco.mj_forward(model, kin_data)
ee_id = model.body("wrist_3_link").id
Jp = np.zeros((3, model.nv))
Jr = np.zeros_like(Jp)
dJp = np.zeros_like(Jp)
dJr = np.zeros_like(Jp)
mujoco.mj_jacBody(model, kin_data, Jp, Jr, ee_id)
mujoco.mj_jacDot(model, kin_data, dJp, dJr, kin_data.xpos[ee_id], ee_id)
R = kin_data.xmat[ee_id].reshape(3, 3)
Jp_local, Jr_local = R.T @ Jp, R.T @ Jr
print("Point velocity, world [m/s]:", Jp @ kin_data.qvel)
print("Point velocity, local [m/s]:", Jp_local @ kin_data.qvel)
np.testing.assert_allclose(R @ (Jp_local @ kin_data.qvel), Jp @ kin_data.qvel, atol=1e-12)

acceleration = np.linspace(0.1, -0.1, model.nv)
point_acceleration = Jp @ acceleration + dJp @ kin_data.qvel
angular_acceleration = Jr @ acceleration + dJr @ kin_data.qvel
eps_time = 1e-5
point_velocities = []
for direction in (-1, 1):
    dt = direction * eps_time
    nearby = mujoco.MjData(model)
    nearby.qpos[:] = kin_data.qpos
    # This second-order path is valid for this hinge-only arm.
    mujoco.mj_integratePos(model, nearby.qpos, kin_data.qvel + 0.5 * dt * acceleration, dt)
    nearby.qvel[:] = kin_data.qvel + dt * acceleration
    mujoco.mj_forward(model, nearby)
    J_nearby = np.zeros_like(Jp)
    mujoco.mj_jacBody(model, nearby, J_nearby, None, ee_id)
    point_velocities.append(J_nearby @ nearby.qvel)
a_finite_difference = (point_velocities[1] - point_velocities[0]) / (2 * eps_time)
np.testing.assert_allclose(point_acceleration, a_finite_difference, atol=1e-8)
print("Point acceleration [m/s²]:", point_acceleration)
print("Angular acceleration [rad/s²]:", angular_acceleration)

### Whole-robot center of mass

The center of mass is the mass-weighted average of **body COM positions**:

$$
r_C=\frac{\sum_i m_i r_i}{\sum_i m_i}.
$$

Its velocity and acceleration satisfy

$$
v_C=J_Cv,
$$

$$
a_C=J_C\dot v+\dot J_Cv.
$$

**MuJoCo quantities**

- `subtree_com`: the COM of a body and its descendants.
- `mj_jacSubtreeCom`: the corresponding COM Jacobian.
- Subtree 0 includes all bodies in this single-robot scene, including fixed-base masses.

**Checks in the next cell:**

1. Compare the stored COM with the mass-weighted sum.
2. Compare Jacobian-based COM velocity with a finite difference of position.
3. Estimate the Jacobian derivative along the velocity and compute COM acceleration.

Finite differences give local numerical estimates, not symbolic expressions.

In [ ]:
J_com = np.zeros((3, model.nv))
mujoco.mj_jacSubtreeCom(model, kin_data, J_com, 0)
com_position = kin_data.subtree_com[0].copy()
weighted_com = (model.body_mass[:, None] * kin_data.xipos).sum(axis=0) / model.body_mass.sum()
np.testing.assert_allclose(com_position, weighted_com, atol=1e-12)
com_velocity = J_com @ kin_data.qvel
J_com_nearby = []
com_nearby = []
for direction in (-1, 1):
    nearby = mujoco.MjData(model)
    nearby.qpos[:] = kin_data.qpos
    mujoco.mj_integratePos(model, nearby.qpos, kin_data.qvel, direction * eps_time)
    mujoco.mj_forward(model, nearby)
    J = np.zeros_like(J_com)
    mujoco.mj_jacSubtreeCom(model, nearby, J, 0)
    J_com_nearby.append(J)
    com_nearby.append(nearby.subtree_com[0].copy())
dJ_com = (J_com_nearby[1] - J_com_nearby[0]) / (2 * eps_time)
np.testing.assert_allclose(com_velocity, (com_nearby[1] - com_nearby[0]) / (2 * eps_time), atol=1e-8)
print("COM position [m]:", com_position)
print("COM velocity [m/s]:", com_velocity)
print("COM acceleration [m/s²]:", J_com @ acceleration + dJ_com @ kin_data.qvel)

## 5. Dynamics and energy

### Inverse dynamics: acceleration to force

Inverse dynamics asks: **which generalized force produces a specified acceleration at this state?**

The force balance is

$$
M(q)\dot v+c(q,v)=
\tau_{\mathrm{passive}}+\tau_{\mathrm{actuator}}+
\tau_{\mathrm{applied}}+\tau_{\mathrm{constraint}}.
$$

MuJoCo returns

$$
\tau_{\mathrm{inverse}}=
M\dot v+c-\tau_{\mathrm{passive}}-\tau_{\mathrm{constraint}}.
$$

**Using `mj_inverse`:**

- Set `qpos`, `qvel`, and the **requested** `qacc`.
- Call the function without stepping time.
- Read the generalized force from `qfrc_inverse`.

This result is not automatically an actuator command; the actuator transmission and force law still matter.

In [ ]:
data.qpos = q0
data.qvel = rng.standard_normal(nv)
data.qacc = rng.standard_normal(nv)
mujoco.mj_inverse(model, data)
combined_force = data.qfrc_inverse.copy()
combined_force

Render the pose used in the inverse-dynamics calculation. An image alone cannot establish whether the required torques are feasible.

In [ ]:
renderer.update_scene(data)
media.show_image(renderer.render())

### Static equilibrium

Set velocity and requested acceleration to zero:

$$
v=0,\qquad \dot v=0.
$$

Inverse dynamics then gives the generalized force required for static equilibrium.

- With no passive or constraint contribution, this equals gravity compensation.
- Contacts, joint limits, or passive forces can change the result.

In [ ]:
data.qvel = np.zeros(6)
data.qacc = np.zeros(6)
mujoco.mj_inverse(model, data)
gravity_force = data.qfrc_inverse.copy()
gravity_force

### Joint-space inertia

`M` is MuJoCo’s compact sparse representation of the joint-space inertia. It is not a dense matrix flattened row by row.

In [ ]:
data.M

**Expand the sparse inertia storage**

For the pinned MuJoCo 3.13 API:

```python
mujoco.mj_fullM(model, data, destination)
```

- `destination` must have shape `(nv, nv)`.
- Refresh the forward or inverse pipeline at the current configuration first.
- Use the dense result for the small numerical examples below.

In [ ]:
inertia_matrix = np.zeros((nv,nv))
mujoco.mj_fullM(model, data, inertia_matrix)

**Check the inertia matrix:**

$$
M=M^T,\qquad \lambda_{\min}(M)>0.
$$

Symmetry and positive definiteness are expected for independent, nondegenerate degrees of freedom. Inspect the matrix and its eigenvalues.

In [ ]:
print(inertia_matrix)
np.testing.assert_allclose(inertia_matrix, inertia_matrix.T, atol=1e-12)
print("Inertia eigenvalues:", np.linalg.eigvalsh(inertia_matrix))

### Separate the terms in the force balance

The generalized equation uses tangent velocities, including for floating bases:

$$
M(q)\dot v+c(q,v)=
\tau_{\mathrm{actuator}}+\tau_{\mathrm{applied}}+
\tau_{\mathrm{passive}}+\tau_{\mathrm{constraint}}.
$$

Following MuJoCo and the lecture notes, $c$ denotes the **full bias force**, combining gravity and velocity-dependent effects. The legacy symbol $h$ denotes this same full bias:

$$
c(q,v)=c_{\mathrm{vel}}(q,v)+g(q).
$$

**Read the MuJoCo fields:**

- `qfrc_bias`: gravity plus Coriolis/centrifugal force, on the left-hand side.
- `qfrc_actuator`: delivered generalized actuator force.
- `qfrc_applied`: externally applied generalized force.
- `qfrc_passive`: contributions such as joint damping and springs.
- `qfrc_constraint`: contributions from contacts, limits, and equality constraints.

All force vectors have `nv` entries; the inertia matrix has shape `(nv, nv)`.

**Separate gravity from velocity-dependent bias:**

$$
g(q)=c(q,0),\qquad c_{\mathrm{vel}}(q,v)=c(q,v)-g(q).
$$

A matrix whose product with velocity equals the Coriolis/centrifugal force is not unique. We therefore work directly with the force vector.

**Next cell:** evaluate the terms, check their residual, and compare acceleration with a dense linear solve. MuJoCo itself uses recursive rigid-body calculations and sparse factorization; the dense solve is a teaching check.

In [ ]:
dyn_data = mujoco.MjData(model)
dyn_data.qpos[:] = q0
dyn_data.qvel[:] = np.linspace(-0.2, 0.3, model.nv)
dyn_data.ctrl[:] = q0  # Current Menagerie actuators are position servos.
mujoco.mj_forward(model, dyn_data)
M = np.empty((model.nv, model.nv))
mujoco.mj_fullM(model, dyn_data, M)
bias = dyn_data.qfrc_bias.copy()
rest_data = mujoco.MjData(model)
rest_data.qpos[:] = dyn_data.qpos
mujoco.mj_forward(model, rest_data)
g = rest_data.qfrc_bias.copy()
velocity_bias = bias - g
forces = (dyn_data.qfrc_actuator + dyn_data.qfrc_applied
          + dyn_data.qfrc_passive + dyn_data.qfrc_constraint)
residual = M @ dyn_data.qacc + bias - forces
np.testing.assert_allclose(residual, 0.0, atol=1e-8)
np.testing.assert_allclose(np.linalg.solve(M, forces - bias), dyn_data.qacc, atol=1e-8)
assert np.linalg.eigvalsh(M).min() > 0
print("Gravity [N·m]:", g)
print("Coriolis/centrifugal force [N·m]:", velocity_bias)
print("Force-balance residual norm:", np.linalg.norm(residual))

### Forward/inverse dynamics consistency

At a consistent forward solution, inverse dynamics should recover

$$
\tau_{\mathrm{inverse}}=\tau_{\mathrm{actuator}}+\tau_{\mathrm{applied}}.
$$

**Next cell:** pass the forward solution's acceleration to `mj_inverse` and compare generalized forces. Allow for constraint-solver tolerance. The result is a force, not a position-servo target.

In [ ]:
expected_inverse = dyn_data.qfrc_actuator.copy() + dyn_data.qfrc_applied
mujoco.mj_inverse(model, dyn_data)
np.testing.assert_allclose(dyn_data.qfrc_inverse, expected_inverse, atol=1e-6)
print("Forward/inverse force difference:", np.linalg.norm(dyn_data.qfrc_inverse - expected_inverse))

### Kinetic energy, potential energy, and the Lagrangian

For this rigid-body arm:

$$
T=\frac12v^TM(q)v,
$$

$$
L=T-V.
$$

- **Kinetic energy:** energy due to motion.
- **Potential energy:** gravitational energy, plus elastic energy if springs are present.
- **Lagrangian:** kinetic minus potential energy.
- **Damping:** dissipative force; it is not part of potential energy.

**MuJoCo calls**

- `mj_energyPos` writes potential energy to `data.energy[0]`.
- `mj_energyVel` writes kinetic energy to `data.energy[1]`.

For this spring-free hinge arm, we also expect

$$
\frac{\partial V}{\partial q}=g(q).
$$

**Next cell:** check kinetic energy against the mass-matrix expression and the potential-energy gradient against gravity.

Potential energy has an arbitrary zero level. Actuation, damping, contact, and numerical integration can change total energy, so a controlled rollout need not conserve it.

In [ ]:
mujoco.mj_forward(model, dyn_data)
mujoco.mj_energyPos(model, dyn_data)
mujoco.mj_energyVel(model, dyn_data)
V, T = dyn_data.energy.copy()
np.testing.assert_allclose(T, 0.5 * dyn_data.qvel @ M @ dyn_data.qvel, atol=1e-10)
L = T - V
energy_gradient = np.zeros(model.nv)
for j in range(model.nv):
    energies = []
    for direction in (-1, 1):
        sample = mujoco.MjData(model)
        sample.qpos[:] = dyn_data.qpos
        mujoco.mj_integratePos(model, sample.qpos, np.eye(model.nv)[j], direction * 1e-5)
        mujoco.mj_forward(model, sample)
        mujoco.mj_energyPos(model, sample)
        energies.append(sample.energy[0])
    energy_gradient[j] = (energies[1] - energies[0]) / (2e-5)
np.testing.assert_allclose(energy_gradient, g, atol=1e-6)
print(f"T={T:.6f} J, V={V:.6f} J, L={L:.6f} J")

### Applied forces and wrenches: virtual work

A Cartesian force at a body-fixed point and a torque about that point produce

$$
\tau_{\mathrm{external}}=J_p^TF+J_r^TN.
$$

The virtual-work power identity is

$$
v^T\tau_{\mathrm{external}}=(J_pv)^TF+(J_rv)^TN.
$$

**Keep conventions consistent:**

- Jacobians and forces must use matching frames and origins.
- `mj_applyFT` takes force and torque in **world axes**.
- Its point argument is a **world position**.
- A point force carries no independent torque; a wrench includes both force and torque.

**Next cell:** map a prescribed wrench to generalized force, verify the power identity, and apply the result through `qfrc_applied`.

The applied force enters the right-hand side of the dynamics with a plus sign. Clear `qfrc_applied` when removing the load. This force is prescribed externally; it is not a contact reaction solved by MuJoCo.

In [ ]:
force_world = np.array([1.0, -0.5, 2.0])
torque_world = np.array([0.1, 0.0, -0.2])
tau_wrench = np.zeros(model.nv)
mujoco.mj_applyFT(model, kin_data, force_world, torque_world,
                  kin_data.xpos[ee_id], ee_id, tau_wrench)
np.testing.assert_allclose(tau_wrench, Jp.T @ force_world + Jr.T @ torque_world, atol=1e-12)
power_joint = kin_data.qvel @ tau_wrench
power_cartesian = (Jp @ kin_data.qvel) @ force_world + (Jr @ kin_data.qvel) @ torque_world
np.testing.assert_allclose(power_joint, power_cartesian, atol=1e-12)
kin_data.qfrc_applied[:] = tau_wrench
mujoco.mj_forward(model, kin_data)
print("Generalized external force [N·m]:", tau_wrench)
print("External mechanical power [W]:", power_joint)
kin_data.qfrc_applied[:] = 0.0

## 6. Floating-base robots: load G1 and Go2

Before discussing legged-robot contacts, load and **look at both robots**.

- **G1:** a humanoid with two feet, a free pelvis, and articulated arms and legs.
- **Go2:** a quadruped with a free base and four articulated legs.
- `mm.load(...)` downloads each model once and reuses its cache afterward.

**Next cell:** create independent model/state pairs, select the provided initial keyframes, and render each robot. These are model poses, not demonstrations of balance. No time integration or balance controller runs here.

Each temporary renderer is closed after its image is displayed.

In [ ]:
g1_model = mm.load("unitree_g1")
go2_model = mm.load("unitree_go2")
g1_data = mujoco.MjData(g1_model)
go2_data = mujoco.MjData(go2_model)
legged_robots = {"G1": (g1_model, g1_data), "Go2": (go2_model, go2_data)}
legged_previews = {}
for name, (robot_model, robot_data) in legged_robots.items():
    if robot_model.nkey:
        mujoco.mj_resetDataKeyframe(robot_model, robot_data, 0)
    mujoco.mj_forward(robot_model, robot_data)
    camera = mujoco.MjvCamera()
    camera.lookat[:] = robot_data.subtree_com[0]
    camera.distance = 2.0 if name == "G1" else 1.5
    camera.azimuth, camera.elevation = 135, -15
    with mujoco.Renderer(robot_model, width=640, height=480) as robot_renderer:
        robot_renderer.update_scene(robot_data, camera=camera)
        legged_previews[name] = robot_renderer.render().copy()
        print(f"{name}: nq={robot_model.nq}, nv={robot_model.nv}, nu={robot_model.nu}")
        media.show_image(legged_previews[name])

### Floating-base coordinates and quaternion differences

A free joint stores **three translations plus a four-component quaternion**, but its velocity has **three linear plus three angular components**.

With one free joint and otherwise hinge joints:

$$
n_q=n_v+1.
$$

For these models, six base velocity coordinates have no direct joint actuator. Gravity, internal coupling, and contact can still accelerate the base.

**Inspect rather than assume:**

1. Find each model's free joint and its position/velocity addresses.
2. Print the joint and control dimensions.
3. On Go2, integrate a small tangent velocity with `mj_integratePos`.
4. Recover it with `mj_differentiatePos` and check quaternion normalization.

Do not add a velocity vector directly to `qpos`. The dimensions and quaternion geometry differ.

In [ ]:
for name, (robot_model, robot_data) in legged_robots.items():
    free_ids = np.flatnonzero(robot_model.jnt_type == mujoco.mjtJoint.mjJNT_FREE)
    assert len(free_ids) == 1
    root_joint = int(free_ids[0])
    print(name, "free-joint position/velocity addresses:",
          robot_model.jnt_qposadr[root_joint], robot_model.jnt_dofadr[root_joint])
    print("  Stored positions, tangent velocities, actuators:",
          robot_model.nq, robot_model.nv, robot_model.nu)
    assert robot_model.nq == robot_model.nv + 1
    assert robot_model.nv - robot_model.nu == 6

root_joint = int(np.flatnonzero(go2_model.jnt_type == mujoco.mjtJoint.mjJNT_FREE)[0])
root_qadr = go2_model.jnt_qposadr[root_joint]
q_reference = go2_data.qpos.copy()
velocity = np.linspace(-0.1, 0.1, go2_model.nv)
q_displaced = q_reference.copy()
mujoco.mj_integratePos(go2_model, q_displaced, velocity, 0.01)
recovered_velocity = np.zeros(go2_model.nv)
mujoco.mj_differentiatePos(go2_model, recovered_velocity, 0.01, q_reference, q_displaced)
np.testing.assert_allclose(recovered_velocity, velocity, atol=1e-12)
np.testing.assert_allclose(np.linalg.norm(q_displaced[root_qadr + 3:root_qadr + 7]), 1.0, atol=1e-12)

### Contact geometry, frames, and friction

Use the **same loaded G1 and Go2 scenes** to inspect ground contacts. `mj_forward` evaluates reactions at the selected pose; it does not guarantee that the robot is in static equilibrium.

**Contact data**

- `contact.geom`: the two collision geoms, which identify the interacting bodies.
- `contact.pos` and `contact.frame`: world position and contact axes.
- `contact.dim`: number of force/torque components permitted by the contact.
- `mj_contactForce`: `[force, torque]` in contact axes, with the normal first.

A contact with `condim=3` supports normal and sliding forces. Dimensions 4 and 6 also include torsional and rolling friction. Read the compiled contact's dimension: different parts of one robot can use different settings.

For an elliptic cone, the scaled tangential force and allowed friction torques satisfy

$$
\left\|\left[\frac{f_{t1}}{\mu_1},\frac{f_{t2}}{\mu_2},
\frac{\tau_n}{\mu_3},\frac{\tau_{t1}}{\mu_4},\frac{\tau_{t2}}{\mu_5}\right]\right\|_2\le f_n.
$$

Include only the components permitted by the contact dimension. Torsional/rolling coefficients include a length scale. Pyramidal cones use a different, polyhedral approximation; do not test them with this elliptic bound.

**Next cell:** inspect contacts on both robots, test Go2's elliptic cone where applicable, and verify each model's full force balance. Joint limits and friction loss can also contribute to `qfrc_constraint`.

In [ ]:
for name, (robot_model, robot_data) in legged_robots.items():
    mujoco.mj_forward(robot_model, robot_data)
    assert robot_data.ncon > 0, f"No contacts in the initial {name} pose"
    print(f"{name}: {robot_data.ncon} contacts; cone={robot_model.opt.cone}")
    for contact_id in range(robot_data.ncon):
        contact = robot_data.contact[contact_id]
        wrench = np.zeros(6)
        mujoco.mj_contactForce(robot_model, robot_data, contact_id, wrench)
        names = [robot_model.geom(int(g)).name or f"geom_{g}" for g in contact.geom]
        print(" ", names, "dim:", contact.dim, "normal force [N]:", round(wrench[0], 3))
        assert wrench[0] >= -1e-7
        if robot_model.opt.cone == mujoco.mjtCone.mjCONE_ELLIPTIC and contact.dim > 1:
            coefficients = contact.friction[:contact.dim - 1]
            assert np.all(coefficients > 0)
            scaled_friction = wrench[1:contact.dim] / coefficients
            assert np.linalg.norm(scaled_friction) <= wrench[0] + 1e-5
    robot_M = np.zeros((robot_model.nv, robot_model.nv))
    mujoco.mj_fullM(robot_model, robot_data, robot_M)
    residual = (robot_M @ robot_data.qacc + robot_data.qfrc_bias
                - robot_data.qfrc_passive - robot_data.qfrc_actuator
                - robot_data.qfrc_applied - robot_data.qfrc_constraint)
    np.testing.assert_allclose(residual, 0.0, atol=1e-6)
    print(" Force-balance residual:", np.linalg.norm(residual))

## 7. Time integration: passive Go2 motion

A model can be simulated without designing a controller. Here we load a separate Go2 instance and **disable actuation** so gravity, passive joint forces, and contacts determine its motion.

**Integration and rendering are different:**

- `model.opt.timestep` sets the physical step duration.
- `mj_step` advances the state and simulation clock.
- `mj_forward` refreshes derived quantities at the resulting state.
- The video frame rate only determines image sampling.

The robot can collapse or fall without support from its actuators. This example demonstrates simulation and contact, not standing stability.

**Next cell:** simulate 0.4 seconds, record a short video, and check finite state and the final force balance. When comparing timesteps, keep the final physical duration fixed.

Contact solving is numerical. We use at most 100 solver iterations and a `1e-10` stopping tolerance for the force-balance check; finite solver error means the residual need not be machine zero.

In [ ]:
passive_model = mm.load("unitree_go2")
passive_model.opt.iterations = 100
passive_model.opt.tolerance = 1e-10
passive_model.opt.disableflags |= mujoco.mjtDisableBit.mjDSBL_ACTUATION
passive_data = mujoco.MjData(passive_model)
if passive_model.nkey:
    mujoco.mj_resetDataKeyframe(passive_model, passive_data, 0)
passive_data.ctrl[:] = 0
mujoco.mj_forward(passive_model, passive_data)
camera = mujoco.MjvCamera()
camera.lookat[:] = passive_data.subtree_com[0]
camera.distance, camera.azimuth, camera.elevation = 1.5, 135, -20
frames = []
duration, framerate = 0.4, 30
steps = round(duration / passive_model.opt.timestep)
with mujoco.Renderer(passive_model, width=640, height=480) as passive_renderer:
    for step in range(steps):
        mujoco.mj_step(passive_model, passive_data)
        assert np.isfinite(passive_data.qpos).all() and np.isfinite(passive_data.qvel).all()
        if len(frames) < passive_data.time * framerate:
            mujoco.mj_forward(passive_model, passive_data)
            passive_renderer.update_scene(passive_data, camera=camera)
            frames.append(passive_renderer.render().copy())
media.show_video(frames, fps=framerate)
mujoco.mj_forward(passive_model, passive_data)
np.testing.assert_allclose(passive_data.qfrc_actuator, 0.0, atol=1e-12)
passive_M = np.zeros((passive_model.nv, passive_model.nv))
mujoco.mj_fullM(passive_model, passive_data, passive_M)
residual = (passive_M @ passive_data.qacc + passive_data.qfrc_bias
            - passive_data.qfrc_passive - passive_data.qfrc_applied - passive_data.qfrc_constraint)
np.testing.assert_allclose(residual, 0.0, atol=1e-6)
print("Final time [s], contacts:", passive_data.time, passive_data.ncon)

### Release graphics resources

Close the offscreen renderer when the simulation experiments are complete. Rerun the initial preview cell if you want to draw with it again.

In [ ]:
renderer.close()
renderer = None

## 8. Questions and controlled experiments

1. Compare UR5e, G1, and Go2 dimensions. Why do floating bases have one more stored position than velocity coordinate?
2. Recover a Go2 configuration change with `mj_differentiatePos`. Why is ordinary subtraction unsuitable for quaternions?
3. Compare body origins and centers of mass, then express the same point velocity in local and world axes.
4. Verify the acceleration formula using the Jacobian and its derivative.
5. At fixed configuration, distinguish gravity, velocity-dependent bias, and passive forces. Recheck the force balance.
6. Compare kinetic energy with the mass-matrix expression and potential-energy derivatives with gravity.
7. Inspect G1 and Go2 contact geom names, dimensions, cone types, and normal forces. Why does a keyframe not guarantee equilibrium?
8. Halve the passive Go2 timestep at fixed duration and compare the final pose and contact forces.

Change one quantity at a time, record units, and report numerical tolerances.

## 9. Next steps: edit the model with `MjSpec`

You can modify a robot's structure and parameters, not just its simulated state.

### Three objects, three purposes

- **`MjSpec`:** an editable model description. Change joints, geoms, sites, actuators, or bodies here.
- **`MjModel`:** the compiled model used by the simulator.
- **`MjData`:** the state and workspace allocated for a particular compiled model.

### A small editing example

The next cell loads a fresh UR5e specification and changes one joint's damping:

1. Obtain an editable specification from Menagerie.
2. Find the first joint by name and edit `damping[0]`, its linear damping coefficient in MuJoCo 3.13.
3. Compile a new model and allocate matching state data.
4. Verify that the compiled parameter has the requested value.

This edits an in-memory specification; it does not overwrite the cached source files or the model used in earlier experiments. For structural edits, compile again and create matching data and rendering resources.

In [ ]:
edited_spec = mm.get("universal_robots_ur5e").spec()
joint_name = model.joint(0).name
edited_spec.joint(joint_name).damping[0] = 1.0  # Linear damping coefficient.
edited_model = edited_spec.compile()
edited_data = mujoco.MjData(edited_model)
mujoco.mj_forward(edited_model, edited_data)
edited_dof = edited_model.jnt_dofadr[edited_model.joint(joint_name).id]
np.testing.assert_allclose(edited_model.dof_damping[edited_dof], 1.0)
print(f"Edited {joint_name}: damping = {edited_model.dof_damping[edited_dof]} N·m·s/rad")

### Documentation to explore next

**Model construction and editing**

- [Python `MjSpec` model editing](https://mujoco.readthedocs.io/en/stable/python.html#model-editing) — Python examples for constructing, editing, and compiling specifications.
- [Model Editing guide](https://mujoco.readthedocs.io/en/stable/programming/modeledit.html) — the specification lifecycle and structural editing API.
- [MJCF XML reference](https://mujoco.readthedocs.io/en/stable/XMLreference.html) — model elements and attributes, including joints, geoms, actuators, sensors, and simulation options.
- [Modeling guide](https://mujoco.readthedocs.io/en/stable/modeling.html) — how model choices affect physical behavior.
- [Menagerie Python API](https://github.com/google-deepmind/mujoco_menagerie/blob/main/python/DOC.md) — editable robot specifications, model files, and caching.

**Equations and function calls**

- [MuJoCo computation](https://mujoco.readthedocs.io/en/stable/computation/index.html) — dynamics, actuation, and constraint formulations.
- [API function reference](https://mujoco.readthedocs.io/en/stable/APIreference/APIfunctions.html) — the functions used throughout this practice.

**Try next:** change one joint's damping, add a tool-frame site, or adjust a contact material. Compile the modified specification, compare it with the baseline, and explain the observed effect.

The course pins MuJoCo 3.13.0. If the stable documentation describes a newer release, select the matching version in the documentation version selector.